In [2]:
#模型下载
from modelscope import snapshot_download
model_dir = snapshot_download('ZhipuAI/glm-4-voice-decoder', local_dir='./THUDM/glm-4-voice-decoder')

2024-11-14 10:36:21,668 - modelscope - INFO - PyTorch version 2.0.1+cu118 Found.
2024-11-14 10:36:21,669 - modelscope - INFO - Loading ast index from /root/.cache/modelscope/ast_indexer
2024-11-14 10:36:21,788 - modelscope - INFO - Loading done! Current index file version is 1.15.0, with md5 37b831191a47f279b705f90597489b75 and a total number of 980 components indexed


transformer is not installed, please install it if you want to use related modules


Downloading: 100%|██████████| 4.34k/4.34k [00:00<00:00, 9.53kB/s]
Downloading: 100%|██████████| 48.0/48.0 [00:00<00:00, 179B/s]
Downloading: 100%|██████████| 425M/425M [00:07<00:00, 62.6MB/s] 
Downloading: 100%|██████████| 78.1M/78.1M [00:01<00:00, 43.6MB/s]
Downloading: 100%|██████████| 6.34k/6.34k [00:00<00:00, 25.0kB/s]
Downloading: 100%|██████████| 1.38k/1.38k [00:00<00:00, 5.59kB/s]


In [ ]:
# model_dir = snapshot_download('ZhipuAI/glm-4-voice-9b', local_dir='./THUDM/glm-4-voice-9b')

In [3]:
model_dir = snapshot_download('ZhipuAI/glm-4-voice-tokenizer', local_dir='./THUDM/glm-4-voice-tokenizer')

Downloading: 100%|██████████| 1.71k/1.71k [00:00<00:00, 4.74kB/s]
Downloading: 100%|██████████| 57.0/57.0 [00:00<00:00, 164B/s]
Downloading: 100%|██████████| 6.34k/6.34k [00:00<00:00, 13.1kB/s]
Downloading: 100%|██████████| 1.36G/1.36G [00:17<00:00, 83.1MB/s]
Downloading: 100%|██████████| 340/340 [00:00<00:00, 1.27kB/s]
Downloading: 100%|██████████| 1.27k/1.27k [00:00<00:00, 5.49kB/s]


In [1]:
from flow_inference import AudioDecoder
import os.path


flow_path="THUDM/glm-4-voice-decoder"
tokenizer_path="THUDM/glm-4-voice-tokenizer"

flow_config = os.path.join(flow_path, "config.yaml")
flow_checkpoint = os.path.join(flow_path, "flow.pt")
hift_checkpoint = os.path.join(flow_path, "hift.pt")
device = "cuda"
audio_decoder: AudioDecoder = None
whisper_model, feature_extractor = None, None

In [2]:

from speech_tokenizer.modeling_whisper import WhisperVQEncoder

# Speech tokenizer
whisper_model = (WhisperVQEncoder.from_pretrained(tokenizer_path).eval().to(device))


/root/miniconda3/envs/GLM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained(tokenizer_path)


In [4]:

import sys

sys.path.insert(0, "./cosyvoice")
sys.path.insert(0, "./third_party/Matcha-TTS")

# Flow & Hift
audio_decoder = AudioDecoder(config_path=flow_config,
                             flow_ckpt_path=flow_checkpoint,
                             hift_ckpt_path=hift_checkpoint,
                             device=device,)


/root/miniconda3/envs/GLM/lib/python3.10/site-packages/torch/_jit_internal.py:739: FutureWarning: ignore(True) has been deprecated. TorchScript will now drop the function call on compilation. Use torch.jit.unused now. {}
  warnings.warn(
/root/miniconda3/envs/GLM/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
/root/miniconda3/envs/GLM/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:28: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


In [5]:
# audio_path = '/home/rickc/Projects/LLMs/CosyVoice/origin_32k.wav'

audio_path = '/home/rickc/Projects/LLMs/SpeechLLM/sample_audio_1_32k.wav'

In [6]:
import IPython.display as ipd


sr = 32000

# 在 Jupyter Notebook 中播放音频
ipd.Audio(audio_path, rate=sr)

In [7]:
import torchaudio

wav, sr = torchaudio.load(audio_path)
wav.shape

torch.Size([1, 123776])

In [8]:
123776/32000

3.868

In [9]:
3.868*12.5

48.35

In [13]:

from speech_tokenizer.utils import extract_speech_token

# 提取音频中的 token
audio_tokens = extract_speech_token(whisper_model, feature_extractor, [audio_path])[0]
len(audio_tokens)

49

In [21]:
len(audio_tokens)

60

In [22]:

import uuid
import torch


# 生成唯一标识符
this_uuid = str(uuid.uuid4())



# 生成语音和梅尔特征
with torch.no_grad():
    # 转换为张量
    tts_token = torch.tensor(audio_tokens, device=device).unsqueeze(0)
    tts_token
    tts_speech, tts_mel = audio_decoder.token2wav(
        tts_token,
        uuid=this_uuid,
    )

tts_speech.shape

/root/miniconda3/envs/GLM/lib/python3.10/site-packages/torch/nn/modules/conv.py:306: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv1d(input, weight, bias, self.stride,


torch.Size([1, 105728])

In [12]:
77312/22050

3.5062131519274375

In [13]:
tts_token.dtype

torch.int64

In [14]:
tts_token.shape

torch.Size([1, 49])

In [23]:
import IPython.display as ipd

# 假设 audio_data 是你的音频数据，sr 是采样率
# audio_data = np.array(new_dataset['mp3'][0]['array'])
sr = 22050 
tts_speech = tts_speech.cpu()
# 在 Jupyter Notebook 中播放音频
ipd.Audio(tts_speech, rate=sr)

#  '云南新疆地区虽有减产，但周渡产量数据依旧处于同期高位五月份，四川地区进入平水期，六月份云南地区进入分水期，相应电价又将有所回落。'